In [ ]:
from IPython.display import HTML


In [ ]:
# Helper function: plot where the model puts high probability gates.
def plot_gate_probs(txt, gate_probs):
    
    # Ensure gate_probs and input_text have the same length
    gate_probs = gate_probs[:len(txt)]
    
    # Create HTML with colored text based on probabilities
    colored_text = ""
    colorbar = ""
    
    # Create a colorbar showing the gradient
    for i in range(11):  # 0.0 to 1.0 in steps of 0.1
        prob = i / 10
        r = min(1.0, prob)
        b = max(0.0, 1.0 - prob)
        color = f"rgb({int(r*255)}, 0, {int(b*255)})"
        white = f"rgb(255, 255, 255)"
        colorbar += f'<span style="color:{white}; background-color:{color}; margin-right:2px; padding:0 5px;">{prob:.1f}</span>'
    
    # Add a legend for the colorbar
    colorbar_html = f'''
    <div style="margin-bottom:10px;">
        <div style="font-family:monospace; font-size:12px; margin-bottom:3px;">Probability: Low → High</div>
        <div style="font-family:monospace; font-size:14px;">{colorbar}</div>
    </div>
    '''
    
    # Process the text with colors
    for char, prob in zip(txt, gate_probs):
        char = char.replace("<", "&lt")
        char = char.replace(">", "&gt")
        # Convert probability to color (blue->red)
        r = min(1.0, prob)  # Red increases with probability
        b = max(0.0, 1.0 - prob)  # Blue decreases with probability
        color = f"rgb({int(r*255)}, 0, {int(b*255)})"
        # Add the colored character to the output
        colored_text += f'<span style="color:{white}; background-color:{color};">{char}</span>'
    
    # Display the colorbar and colored text
    display(HTML(f'''
    <div>
        {colorbar_html}
        <div style="font-family:monospace; font-size:14px;">{colored_text}</div>
    </div>
    '''))


In [3]:
from clean_code.flexible_bitter_llm import FlexibleBitterLLM, LinearGater, text_to_tensor, per_token_losses_backbone
from clean_code.conditional_sequential import OptimizedSequentialyDependentLinearGater, ScaledSequentialyDependentLinearGater
from accelerate import Accelerator
from training_random_base_model.hparam_utils import get_model_kwargs
from transformers import AutoTokenizer
import torch
from torch.utils.data import DataLoader
from data_processing import split_fineweb


In [17]:
byte_tokenizer = AutoTokenizer.from_pretrained("evabyte/EvaByte", trust_remote_code=True)
learned_checkpoint_path = "training_random_base_model/checkpoints/130M_2025.09.26_04.22"
learned_model_kwargs = get_model_kwargs("130M")
learned_model_kwargs["vocab_size"] = len(byte_tokenizer) 
learned_model_kwargs["GaterClass"] = ScaledSequentialyDependentLinearGater

device = torch.device("cuda")

learned_model = FlexibleBitterLLM(**learned_model_kwargs).to(device, dtype=torch.bfloat16)

accelerator = Accelerator()
learned_model = accelerator.prepare(learned_model)
accelerator.load_state(learned_checkpoint_path)


using self._attn_implementation='flash_attention_2'


In [18]:
_, _, test_set = split_fineweb.get_splits()


Loading dataset from disk:   0%|          | 0/549 [00:00<?, ?it/s]

In [19]:
test_set[5]

{'text': 'Women in face veils detained as France enforces ban\nAt least two women have been briefly detained in France while wearing Islamic veils, after a law banning the garment in public came into force.\nPolice said they were held not because of their veils but for joining an unauthorised protest, and they were later released.\nFrance is the first country in Europe to publicly ban a form of dress some Muslims regard as a religious duty.\nOffenders face a fine of 150 euros (£133; $217) and a citizenship course.\nPeople forcing women to wear the veil face a much larger fine and a prison sentence of up to two years.\nThe two women detained had taken part in a demonstration outside Notre Dame cathedral in Paris. Police said the protest had not been authorised and so people were asked to move on. When they did not, they were arrested.\nGavin Hewitt\'s Europe\nThe law is likely to be largely symbolic... It will be difficult to prove that a woman is being forced to wear a niqab because of

In [20]:
tokens, _ = text_to_tensor(test_set[1], byte_tokenizer, 4096, "cuda")
tokens

tensor([[  1, 139, 165,  ..., 183, 110,  96]], device='cuda:0')

In [21]:
token_list = [byte_tokenizer.decode(t) for t in tokens[0]]
token_list


['<bos>',
 'K',
 'e',
 'n',
 't',
 'i',
 'c',
 'o',
 ' ',
 'C',
 'M',
 'S',
 ' ',
 '8',
 ' ',
 'S',
 'm',
 'a',
 'r',
 't',
 ' ',
 'S',
 'e',
 'a',
 'r',
 'c',
 'h',
 ':',
 ' ',
 'L',
 'a',
 't',
 'e',
 's',
 't',
 ' ',
 't',
 'e',
 'c',
 'h',
 'n',
 'o',
 'l',
 'o',
 'g',
 'i',
 'e',
 's',
 ',',
 ' ',
 'm',
 'o',
 'r',
 'e',
 ' ',
 'b',
 'u',
 'i',
 'l',
 't',
 '-',
 'i',
 'n',
 ' ',
 'f',
 'u',
 'n',
 'c',
 't',
 'i',
 'o',
 'n',
 's',
 ',',
 ' ',
 'm',
 'o',
 'r',
 'e',
 ' ',
 'e',
 'x',
 't',
 'e',
 'n',
 's',
 'i',
 'b',
 'i',
 'l',
 'i',
 't',
 'y',
 '\n',
 'A',
 'r',
 'e',
 ' ',
 'y',
 'o',
 'u',
 ' ',
 'l',
 'o',
 'o',
 'k',
 'i',
 'n',
 'g',
 ' ',
 'f',
 'o',
 'r',
 ' ',
 'a',
 ' ',
 'w',
 'a',
 'y',
 ' ',
 'h',
 'o',
 'w',
 ' ',
 't',
 'o',
 ' ',
 'e',
 'x',
 't',
 'e',
 'n',
 'd',
 ' ',
 'y',
 'o',
 'u',
 'r',
 ' ',
 's',
 'e',
 'a',
 'r',
 'c',
 'h',
 ' ',
 'i',
 'n',
 'd',
 'e',
 'x',
 'e',
 's',
 ' ',
 'w',
 'i',
 't',
 'h',
 ' ',
 'a',
 ' ',
 's',
 'y',
 'n',
 'o',
 'n',

In [24]:
out = learned_model(tokens)
dgp =  out["down_gate_probs"]
plot_gate_probs(token_list, out["down_gate_probs"][0])

TypeError: get_gemma2_attention_mask() got an unexpected keyword argument 'attn_impleemntation'

In [16]:
plot_gate_probs(token_list, out["down_gate_samples"][0])

NameError: name 'out' is not defined

In [64]:
from clean_code.conditional_sequential import ScaledSequentialyDependentLinearGater
from optimizing_code.triton_scan_simple import torch_scan_simple, compute_probs, torch_scan_simple_scaled, compute_probs_scaled
import math
from torch import nn

class PatchedScaledSequentialyDependentLinearGater(ScaledSequentialyDependentLinearGater):
    def __init__(self, embedding_dim: int, downsample_rate: float, filter_size: int = 4):
        super().__init__()
        self.embedding_dim = embedding_dim
        self.downsample_rate = downsample_rate
        self.filter_size = filter_size
        self.filter_layer = nn.Linear(embedding_dim, filter_size)


    def forward(self, x: torch.Tensor, downsample_rate: float) -> torch.Tensor:
        
        if downsample_rate is None:
            downsample_rate = self.downsample_rate

        bias = math.log(downsample_rate / (1 - downsample_rate))
        scale = 1/8.
        
        V = self.filter_layer(x)
        filter_flattened = V.flatten(end_dim=1).detach().to(device="cpu", dtype=torch.float32)
        print(filter_flattened.shape)
        print(f"{bias=}")
        filter_averages = filter_flattened.mean(dim=0)
        filter_stds = filter_flattened.std(dim=0)
        filter_sters = filter_stds / math.sqrt(filter_flattened.shape[0])
        filter_sters = filter_sters.numpy()
        filter_averages = filter_averages.numpy()
        for stder, averge in zip(filter_sters, filter_averages):
            print(f"{averge:.2f} +- {stder:.2f}, ", end="")
        print()
        scan_logits, scan_probs, a = torch_scan_simple_scaled(V, scale_factor=scale, bias=bias)
        logits, probs = compute_probs_scaled(a, V, scale_factor=scale, bias=bias)
        return logits.unsqueeze(-1), probs.unsqueeze(-1), a.unsqueeze(-1)


In [65]:
learned_model.down_layer_gate.__class__ = PatchedScaledSequentialyDependentLinearGater

In [60]:
dgp[0, 1:-1]

tensor([0.4121, 0.4141, 0.4141,  ..., 0.4121, 0.2812, 0.1514], device='cuda:0',
       dtype=torch.bfloat16, grad_fn=<SliceBackward0>)